# ML Baseline — Weather Forecasting for Spanish SYNOP Stations

This notebook summarises the machine learning baseline results for short-range weather forecasting over the Spanish SYNOP station network. It documents model selection, validation methodology, and key findings that motivate the subsequent deep learning phase.

**Forecast targets:** daily maximum temperature (tmax), minimum temperature (tmin), and precipitation (prec)  
**Forecast horizons:** t+1, t+3, t+7 days  
**Prototype stations:** Córdoba, Burgos, Navacerrada, Palma (Puerto)  
**Period:** 1991–present (~30 years, filtered per WMO quality criteria)

## 1. Models and Validation Setup

Two gradient boosting models were evaluated:

- **LightGBM** — histogram-based gradient boosting, faster training
- **XGBoost** — exact greedy gradient boosting

Random Forest was tested on two stations (Córdoba, Burgos) and discarded due to systematically higher MAE and training times 8–10× longer than gradient boosting models.

**Validation scheme:** two complementary schemes were used.

- **Expanding Window CV (20 splits):** training set grows one year at a time; evaluation on the following year. This is the primary scheme, as it respects the temporal structure of climate data and maximises the number of evaluation windows.
- **Blocked CV (9 splits):** fixed-size non-overlapping blocks. Used as a secondary check for stability.

**Stationarity check:** before applying expanding window CV, Augmented Dickey-Fuller (ADF) tests were run on tmax, tmin, and prec for all four prototype stations. All series rejected the unit root hypothesis (p < 0.001), confirming stationarity over the study period. This supports the use of expanding window CV: early training windows remain statistically informative for later evaluation periods.

| Station | Variable | ADF Statistic | p-value |
|---|---|---|---|
| Burgos | tmax | -8.22 | 2.2e-11 |
| Burgos | tmin | -7.52 | 9.1e-10 |
| Burgos | prec | -32.57 | ~0.0 |
| Córdoba | tmax | -8.33 | 1.2e-11 |
| Córdoba | tmin | -7.58 | 6.6e-10 |
| Córdoba | prec | -15.37 | 1.5e-22 |
| Navacerrada | tmax | -9.02 | 3.3e-13 |
| Navacerrada | tmin | -8.24 | 1.9e-11 |
| Navacerrada | prec | -14.37 | 5.4e-22 |
| Palma (Puerto) | tmax | -8.96 | 4.5e-13 |
| Palma (Puerto) | tmin | -9.04 | 2.9e-13 |
| Palma (Puerto) | prec | -26.64 | ~0.0 |

## 2. Results

All metrics are averaged across expanding window CV splits (20 windows).

### 2.1 tmax (°C)

In [1]:
import pandas as pd

tmax_results = [
    # station, model, t+1 MAE, t+1 RMSE, t+1 R², t+3 MAE, t+3 RMSE, t+3 R², t+7 MAE, t+7 RMSE, t+7 R²
    ('Córdoba',     'LightGBM', 1.646, 2.126, 0.940, 2.760, 3.442, 0.835, 3.140, 3.927, 0.787),
    ('Córdoba',     'XGBoost',  1.674, 2.153, 0.938, 2.652, 3.313, 0.854, 2.940, 3.675, 0.820),
    ('Burgos',      'LightGBM', 2.212, 2.801, 0.891, 3.319, 4.082, 0.770, 3.596, 4.431, 0.730),
    ('Burgos',      'XGBoost',  2.247, 2.840, 0.888, 3.307, 4.074, 0.771, 3.584, 4.420, 0.731),
    ('Navacerrada', 'LightGBM', 1.922, 2.440, 0.923, 3.457, 4.291, 0.760, 4.064, 4.959, 0.679),
    ('Navacerrada', 'XGBoost',  1.953, 2.475, 0.920, 3.454, 4.256, 0.765, 3.971, 4.859, 0.694),
    ('Palma',       'LightGBM', 1.366, 1.771, 0.905, 1.754, 2.286, 0.842, 1.901, 2.462, 0.818),
    ('Palma',       'XGBoost',  1.375, 1.780, 0.904, 1.732, 2.259, 0.846, 1.892, 2.455, 0.819),
]

cols = ['Station', 'Model',
        't+1 MAE', 't+1 RMSE', 't+1 R²',
        't+3 MAE', 't+3 RMSE', 't+3 R²',
        't+7 MAE', 't+7 RMSE', 't+7 R²']

df_tmax = pd.DataFrame(tmax_results, columns=cols)
df_tmax.style.format(precision=3).set_caption('tmax — Expanding Window CV (20 splits)')

,Station,Model,t+1 MAE,t+1 RMSE,t+1 R²,t+3 MAE,t+3 RMSE,t+3 R²,t+7 MAE,t+7 RMSE,t+7 R²
0,Córdoba,LightGBM,1.646,2.126,0.940,2.760,3.442,0.835,3.140,3.927,0.787
1,Córdoba,XGBoost,1.674,2.153,0.938,2.652,3.313,0.854,2.940,3.675,0.820
2,Burgos,LightGBM,2.212,2.801,0.891,3.319,4.082,0.770,3.596,4.431,0.730
3,Burgos,XGBoost,2.247,2.840,0.888,3.307,4.074,0.771,3.584,4.420,0.731
4,Navacerrada,LightGBM,1.922,2.440,0.923,3.457,4.291,0.760,4.064,4.959,0.679
5,Navacerrada,XGBoost,1.953,2.475,0.920,3.454,4.256,0.765,3.971,4.859,0.694
6,Palma,LightGBM,1.366,1.771,0.905,1.754,2.286,0.842,1.901,2.462,0.818
7,Palma,XGBoost,1.375,1.780,0.904,1.732,2.259,0.846,1.892,2.455,0.819


### 2.2 tmin (°C)

In [2]:
tmin_results = [
    ('Córdoba',     'LightGBM', 1.437, 1.869, 0.915, 2.103, 2.689, 0.824, 2.323, 2.944, 0.790),
    ('Córdoba',     'XGBoost',  1.452, 1.882, 0.914, 2.109, 2.691, 0.824, 2.338, 2.952, 0.788),
    ('Burgos',      'LightGBM', 1.698, 2.154, 0.847, 2.482, 3.118, 0.681, 2.689, 3.348, 0.634),
    ('Burgos',      'XGBoost',  1.717, 2.183, 0.843, 2.488, 3.121, 0.680, 2.639, 3.293, 0.646),
    ('Navacerrada', 'LightGBM', 1.472, 1.927, 0.914, 2.770, 3.430, 0.730, 3.231, 3.955, 0.634),
    ('Navacerrada', 'XGBoost',  1.504, 1.961, 0.911, 2.784, 3.441, 0.728, 3.141, 3.847, 0.661),
    ('Palma',       'LightGBM', 1.074, 1.417, 0.938, 1.523, 1.960, 0.881, 1.692, 2.165, 0.855),
    ('Palma',       'XGBoost',  1.085, 1.429, 0.937, 1.529, 1.967, 0.881, 1.692, 2.154, 0.857),
]

df_tmin = pd.DataFrame(tmin_results, columns=cols)
df_tmin.style.format(precision=3).set_caption('tmin — Expanding Window CV (20 splits)')

,Station,Model,t+1 MAE,t+1 RMSE,t+1 R²,t+3 MAE,t+3 RMSE,t+3 R²,t+7 MAE,t+7 RMSE,t+7 R²
0,Córdoba,LightGBM,1.437,1.869,0.915,2.103,2.689,0.824,2.323,2.944,0.790
1,Córdoba,XGBoost,1.452,1.882,0.914,2.109,2.691,0.824,2.338,2.952,0.788
2,Burgos,LightGBM,1.698,2.154,0.847,2.482,3.118,0.681,2.689,3.348,0.634
3,Burgos,XGBoost,1.717,2.183,0.843,2.488,3.121,0.680,2.639,3.293,0.646
4,Navacerrada,LightGBM,1.472,1.927,0.914,2.770,3.430,0.730,3.231,3.955,0.634
5,Navacerrada,XGBoost,1.504,1.961,0.911,2.784,3.441,0.728,3.141,3.847,0.661
6,Palma,LightGBM,1.074,1.417,0.938,1.523,1.960,0.881,1.692,2.165,0.855
7,Palma,XGBoost,1.085,1.429,0.937,1.529,1.967,0.881,1.692,2.154,0.857


### 2.3 Precipitation (mm)

In [3]:
prec_results = [
    ('Córdoba',     'LightGBM', 2.184, 4.911, 0.151, 2.476, 5.195, 0.050, 2.579, 5.283, 0.021),
    ('Córdoba',     'XGBoost',  2.174, 4.892, 0.157, 2.483, 5.197, 0.048, 2.547, 5.273, 0.025),
    ('Burgos',      'LightGBM', 1.936, 3.606, 0.134, 2.212, 3.838, 0.021, 2.233, 3.867, 0.009),
    ('Burgos',      'XGBoost',  1.953, 3.622, 0.127, 2.210, 3.832, 0.023, 2.222, 3.852, 0.016),
    ('Navacerrada', 'LightGBM', 4.109, 8.139, 0.201, 4.871, 8.905, 0.046, 5.024, 9.008, 0.025),
    ('Navacerrada', 'XGBoost',  4.126, 8.141, 0.200, 4.892, 8.903, 0.048, 5.029, 9.007, 0.025),
    ('Palma',       'LightGBM', 2.037, 5.150, 0.070, 2.211, 5.314, 0.008, 2.208, 5.297, 0.011),
    ('Palma',       'XGBoost',  2.027, 5.164, 0.064, 2.206, 5.311, 0.009, 2.209, 5.307, 0.006),
]

df_prec = pd.DataFrame(prec_results, columns=cols)
df_prec.style.format(precision=3).set_caption('Precipitation — Expanding Window CV (20 splits)')

,Station,Model,t+1 MAE,t+1 RMSE,t+1 R²,t+3 MAE,t+3 RMSE,t+3 R²,t+7 MAE,t+7 RMSE,t+7 R²
0,Córdoba,LightGBM,2.184,4.911,0.151,2.476,5.195,0.050,2.579,5.283,0.021
1,Córdoba,XGBoost,2.174,4.892,0.157,2.483,5.197,0.048,2.547,5.273,0.025
2,Burgos,LightGBM,1.936,3.606,0.134,2.212,3.838,0.021,2.233,3.867,0.009
3,Burgos,XGBoost,1.953,3.622,0.127,2.210,3.832,0.023,2.222,3.852,0.016
4,Navacerrada,LightGBM,4.109,8.139,0.201,4.871,8.905,0.046,5.024,9.008,0.025
5,Navacerrada,XGBoost,4.126,8.141,0.200,4.892,8.903,0.048,5.029,9.007,0.025
6,Palma,LightGBM,2.037,5.150,0.070,2.211,5.314,0.008,2.208,5.297,0.011
7,Palma,XGBoost,2.027,5.164,0.064,2.206,5.311,0.009,2.209,5.307,0.006


## 3. Model Selection

**LightGBM is selected as the final model for all horizons.**

The decision is based on three consistent findings across all four prototype stations:

1. **LightGBM outperforms XGBoost at t+1** for both tmax and tmin in all four stations, with MAE differences ranging from 0.02°C to 0.03°C.
2. **At t+3 and t+7**, differences between models are small (< 0.10°C) and not supported by statistical significance tests. No clear winner can be established.
3. **LightGBM trains 2–3× faster** than XGBoost across all stations and horizons, which is relevant for scaling to the full 98-station network.

A hybrid approach (LightGBM for t+1, XGBoost for t+3/t+7) was considered but rejected: the marginal performance gain at longer horizons does not justify the added complexity of maintaining two separate model pipelines.

## 4. Precipitation — Negative Result

ML models fail to predict precipitation reliably at any horizon.

- **t+1:** R² between 0.07 and 0.20 across stations. The best result (Navacerrada, 0.20) is the exception, not the norm.
- **t+3 and t+7:** R² collapses to near zero across all stations. The normalised RMSE exceeds 0.97 in most cases, meaning the model barely improves on predicting the climatological mean.

This is not a modelling failure — it reflects the physical nature of precipitation. Daily precipitation is governed by mesoscale and synoptic atmospheric dynamics (frontal systems, convective triggering, orographic effects) that are not captured by local station observations alone. Features available at a single station are insufficient to represent these processes.

This finding motivates two directions explored in subsequent phases:
1. **Deep learning:** LSTM/GRU architectures may capture temporal dependencies in sequential station data that gradient boosting cannot.
2. **Problem reformulation:** if DL does not substantially improve precipitation forecasting, the task will be reformulated as binary classification (rain / no rain) with a physically motivated threshold, which is a more tractable problem and of direct practical value.

## 5. Next Steps

- **Deep Learning phase:** LSTM, GRU and CNN architectures on the same four prototype stations, with the same forecast targets and horizons.
- **Global model (Phase 5):** a single model trained across all 98 SYNOP stations, using latitude, longitude, and altitude as additional features.
- **Precipitation reformulation:** binary classification (rain / no rain) if DL does not show substantial improvement over ML baseline.